# Análisis Operativo de Helpdesk — IT Service Management
## Notebook 1 de 2: Extracción, Transformación y Carga (ETL)

**Autora:** Camila Gámez · Data Analyst  
**Fuente:** Help Desk Tickets · Mendeley Data · CC BY 4.0  
**Citación:** Abdellatif, M. (2025). Help Desk Tickets. Mendeley Data, V2. https://doi.org/10.17632/btm76zndnt.2  
**Período:** enero 2016 – marzo 2023 · **Tickets analizados:** 32,726 (con prioridad clasificada)

---

## 1. Contexto y problema de negocio

Las empresas de servicios tecnológicos operan bajo acuerdos de nivel de servicio (SLA) con sus clientes.
Cada ticket de soporte representa una promesa de respuesta: si se incumple sistemáticamente, el cliente no renueva.

**El problema concreto:** sin métricas operativas estructuradas, el equipo no puede saber si los tickets críticos
se atienden realmente más rápido, qué porcentaje de resoluciones falla, ni dónde se concentra el riesgo
de perder clientes. Ese costo invisible — tickets reabiertos, escalaciones, SLA incumplidos — es el que
este análisis hace visible y cuantificable.

**Pregunta central:**
> *¿Dónde está el riesgo operativo real de este helpdesk, y qué decisiones concretas pueden reducir el costo de atención y proteger la retención de clientes?*

---

## 2. KPIs de la industria ITSM calculados en este proyecto

Estos son los indicadores estándar de **IT Service Management (ITSM)** según el framework ITIL v4.
Cada uno aparece definido y calculado en la sección correspondiente.

| KPI | Definición técnica | Valor obtenido | Dónde se calcula |
|---|---|---|---|
| **MTTR** *(Mean Time to Resolve)* | Tiempo mediano desde apertura hasta cierre | Blocker: 90.6h · Medium: 238.5h | Paso 5 SQL + Paso 6 pandas |
| **FCR** *(First Contact Resolution Rate)* | % tickets resueltos sin cambio de asignado | 57.1% | Paso 5 SQL JOIN |
| **Escalation Rate** | % tickets con más de una reasignación | 42.9% | Paso 5 SQL JOIN |
| **Ticket Churn** *(Reopen Rate)* | % tickets cerrados que volvieron a abrirse | 2.56% global | Paso 4 pandas |
| **Resolution Rate** | % tickets con resolución 'Done' | 93.0% | Paso 3 exploración |
| **Backlog Rate** | % tickets sin fecha de resolución | 1.3% | Paso 3 exploración |
| **SLA Compliance Rate** | % tickets resueltos dentro del umbral de tiempo | Blocker: 22.5% (<8h) | Paso 6 pandas |
| **Ticket Volume YoY** | Crecimiento interanual del volumen | +382% (2018→2022) | Paso 4 pandas |
| **Avg Processing Steps** | Pasos promedio del workflow hasta resolución | 4.0 pasos | Paso 5 SQL |
| **Avg Contributors per Ticket** | Promedio de personas involucradas | Blocker: 1.89 · Medium: 1.30 | Paso 6 pandas |

> **Nota importante para entrevista:** CAC, LTV, ROMI y Churn de clientes son métricas de marketing/SaaS.
> No aplican a operaciones de helpdesk. El término correcto aquí es **Ticket Churn** (Reopen Rate),
> que mide resoluciones fallidas — concepto análogo pero en el dominio de soporte TI.

---

## 3. Datos disponibles

| Archivo | Filas | Columnas | Descripción |
|---|---|---|---|
| `issues.csv` | 66,691 | 58 | Tabla maestra — un registro por ticket con métricas de workflow agregadas |
| `issues_change_history.csv` | 257,508 | 6 | Historial de cada cambio de estado y asignado |
| `issues_snapshot.csv` | 90,963 | 60 | Actividad desagregada por turno de asignación |

**¿Por qué tres tablas y no una?**  
El FCR y Escalation Rate reales solo pueden calcularse cruzando `issues_change_history` con `issues`
mediante JOIN — la tabla principal solo guarda el asignado final, no el historial de reasignaciones.
La estructura relacional replica cómo operan los sistemas ITSM reales (ServiceNow, Jira Service Management).

---

## 4. Mapa completo del proyecto

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NOTEBOOK 1 — ETL (este archivo)          NOTEBOOK 2 — ANÁLISIS Y CONCLUSIONES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PASO 1 · Importaciones y config           PASO 7  · MTTR por prioridad + visual
PASO 2 · Carga de 3 archivos relacionales PASO 8  · Prueba Shapiro-Wilk
PASO 3 · Exploración y diagnóstico        PASO 9  · Prueba Mann-Whitney U
          · Duplicados                    PASO 10 · Ticket Churn por prioridad
          · Nulos por columna             PASO 11 · FCR y Escalation Rate
          · Distribución de variables     PASO 12 · SLA Compliance Rate
          · Resolution Rate y Backlog     PASO 13 · Ticket Volume YoY
          · Rango temporal                PASO 14 · Avg Processing Steps
PASO 4 · Limpieza y transformación        PASO 15 · Externo vs Interno
          · Conversión de fechas          PASO 16 · Correlación Spearman
          · Filtro prioridad unknown      PASO 17 · Segmentación de riesgo compuesto
          · Clasificación ext/interno     PASO 18 · Conclusiones y recomendaciones
          · MTTR en horas
          · Ticket Churn flag
          · Variables temporales
PASO 5 · SQLite + 4 queries SQL
          · MTTR por prioridad (SQL)
          · FCR + Escalation (JOIN SQL)
          · Tendencia mensual (SQL)
          · Carga por turno (SQL)
PASO 6 · KPIs completos en pandas
          · MTTR con percentiles
          · Ticket Churn por prioridad
          · SLA Compliance Rate
          · Avg Contributors
          · Exportación de 6 datasets
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
POWER BI · 3 páginas · medidas DAX · filtros por año y tipo de proyecto
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

---

## 5. Stack tecnológico

| Herramienta | Versión | Rol en el proyecto |
|---|---|---|
| **Python** | 3.11 | Lenguaje principal |
| **pandas** | 2.x | ETL · transformaciones · cálculo de KPIs · exportación |
| **numpy** | 2.x | Operaciones numéricas · percentiles |
| **sqlite3** | stdlib | Réplica local de la BD PostgreSQL original — ejecución de SQL |
| **SQL** | — | 4 queries con JOINs · CASE · JULIANDAY · GROUP BY · SUBSTR |
| **scipy.stats** | — | Shapiro-Wilk (normalidad) + Mann-Whitney U (Notebook 2) |
| **matplotlib + seaborn** | — | Visualizaciones estáticas (Notebook 2) |
| **plotly** | — | Visualizaciones interactivas (Notebook 2) |
| **Power BI Desktop** | — | Dashboard ejecutivo · 3 páginas · medidas DAX |

**ETL vs ELT — contexto de industria:**
- **ETL** (Extract → Transform → Load): patrón de este notebook. Se transforma antes de cargar al destino analítico.
- **ELT** (Extract → Load → Transform): patrón en plataformas cloud como Databricks o BigQuery. Se carga crudo primero y se transforma dentro de la plataforma con SQL o Spark. En Databricks, este pipeline sería un Delta Live Table con las mismas transformaciones.

---

---
# PASO 1 — Importaciones y configuración del entorno

**Qué:** cargar todas las librerías necesarias y definir las rutas del proyecto.  
**Por qué primero:** centralizar importaciones y rutas en una celda hace el proyecto reproducible — cualquier persona puede clonar el repositorio y ejecutar sin modificar nada más que esta sección.  
**Herramientas:** Python · pandas · numpy · sqlite3 · scipy

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

# ─── RUTAS ────────────────────────────────────────────────────────────
# Los CSV están en la carpeta 'Help Desk Tickets', al mismo nivel que el notebook
RUTA_DATOS   = 'Help Desk Tickets/'
RUTA_EXPORTS = 'exports/'
RUTA_SQL     = 'sql/'

os.makedirs(RUTA_EXPORTS, exist_ok=True)
os.makedirs(RUTA_SQL, exist_ok=True)

print(f'pandas {pd.__version__} | numpy {np.__version__}')
print('Entorno configurado.')
print(f'Leyendo datos desde: {os.path.abspath(RUTA_DATOS)}')

pandas 2.3.3 | numpy 2.3.5
Entorno configurado.
Leyendo datos desde: C:\Users\camil\Dropbox\0. DA\Career Accelerator\1. Proyectos\2. Sotware\Help Desk Tickets


---
# PASO 2 — Carga de los tres archivos fuente

**Qué:** leer los tres CSV y verificar que se cargaron correctamente.  
**Por qué tres archivos y no uno:** cada tabla tiene granularidad diferente:
- `issues` = 1 fila por ticket (tabla maestra)
- `issues_change_history` = 1 fila por cada evento de cambio de estado o asignado
- `issues_snapshot` = 1 fila por cada turno de asignación

El JOIN entre `issues_change_history` e `issues` es el único modo de calcular FCR y Escalation Rate reales.  
**Herramienta:** pandas `read_csv`

In [2]:
df_issues   = pd.read_csv(RUTA_DATOS + 'issues.csv',                 low_memory=False)
df_history  = pd.read_csv(RUTA_DATOS + 'issues_change_history.csv',  low_memory=False)
df_snapshot = pd.read_csv(RUTA_DATOS + 'issues_snapshot.csv',        low_memory=False)

print(f'issues.csv                {df_issues.shape[0]:>8,} filas × {df_issues.shape[1]:>3} columnas')
print(f'issues_change_history.csv {df_history.shape[0]:>8,} filas × {df_history.shape[1]:>3} columnas')
print(f'issues_snapshot.csv       {df_snapshot.shape[0]:>8,} filas × {df_snapshot.shape[1]:>3} columnas')

issues.csv                  66,691 filas ×  58 columnas
issues_change_history.csv  257,508 filas ×   6 columnas
issues_snapshot.csv         90,963 filas ×  60 columnas


---
# PASO 3 — Exploración inicial y diagnóstico de calidad

**Qué:** revisar duplicados, nulos, distribución de variables clave, Resolution Rate, Backlog Rate y rango temporal.  
**Por qué antes de limpiar:** explorar primero evita limpiar a ciegas. Cada decisión del Paso 4 nace de lo que se descubre aquí. Saltarse la exploración es la causa más común de análisis con resultados incorrectos.  
**Herramientas:** pandas (`.duplicated()`, `.isnull()`, `.value_counts()`, `.describe()`)

In [3]:
# 3.1 · Duplicados
# Si hubiera filas idénticas, los KPIs quedarían inflados artificialmente
dup_filas = df_issues.duplicated().sum()
dup_ids   = df_issues['id'].duplicated().sum()

print('=== DUPLICADOS ===')
print(f'  Filas completamente duplicadas: {dup_filas}')
print(f'  IDs de ticket duplicados:       {dup_ids}')
print(f'  Diagnóstico: {"Sin duplicados — datos confiables ✓" if dup_filas + dup_ids == 0 else "HAY DUPLICADOS — revisar antes de continuar"}')   

=== DUPLICADOS ===
  Filas completamente duplicadas: 0
  IDs de ticket duplicados:       0
  Diagnóstico: Sin duplicados — datos confiables ✓


In [4]:
# 3.2 · Distribución de variables categóricas clave
# Este paso define el universo del análisis: qué prioridades, tipos y estados existen
# HALLAZGO: el 50.9% de los tickets tiene prioridad 'unknown' — decisión de filtro en Paso 4
for col in ['issue_priority', 'issue_type', 'issue_resolution', 'issue_status']:
    counts = df_issues[col].value_counts()
    pcts   = (counts / len(df_issues) * 100).round(1)
    print(f'\n{col.upper()}:')
    print(pd.DataFrame({'n': counts, '%': pcts}).to_string())


ISSUE_PRIORITY:
                    n     %
issue_priority             
unknown         33965 50.90
Medium          24788 37.20
High             4554  6.80
Highest          2084  3.10
Blocker           656  1.00
Low               560  0.80
Lowest             84  0.10

ISSUE_TYPE:
                    n     %
issue_type                 
Ticket          45275 67.90
Service          5300  7.90
Subtask          4746  7.10
Story            4538  6.80
HD Service       1686  2.50
Task             1540  2.30
Vacation          856  1.30
Project           842  1.30
Sub-task          544  0.80
Epic              403  0.60
Deployment        350  0.50
Retrospective     241  0.40
Sprint Summary    208  0.30
Assistance        109  0.20
Bug                53  0.10

ISSUE_RESOLUTION:
                      n     %
issue_resolution             
Done              62034 93.00
Won't Do           2991  4.50
Duplicate           661  1.00
Cannot Reproduce    152  0.20

ISSUE_STATUS:
                            

In [5]:
# 3.3 · KPI: Resolution Rate y Backlog Rate
# ─────────────────────────────────────────────────────────────────────
# Resolution Rate: % de tickets con resolución 'Done'
#   Mide qué porcentaje del trabajo procesado fue completado exitosamente.
#   En ITSM, un Resolution Rate > 85% se considera saludable.
#
# Backlog Rate: % de tickets sin fecha de resolución
#   Mide la cola de trabajo acumulado sin cerrar.
#   Un Backlog Rate creciente en el tiempo es señal temprana de capacidad insuficiente.

resolution_rate = (df_issues['issue_resolution'] == 'Done').mean() * 100
backlog_rate    = df_issues['issue_resolution_date'].isna().mean() * 100

print(f'Resolution Rate (% tickets con resolución Done): {resolution_rate:.1f}%')
print(f'Backlog Rate    (% tickets sin fecha de cierre): {backlog_rate:.1f}%')
print()
print(f'Interpretación:')
print(f'  · Resolution Rate {resolution_rate:.1f}% > 85% → saludable ✓')
print(f'  · Backlog Rate {backlog_rate:.1f}% → baja acumulación de trabajo pendiente ✓')

Resolution Rate (% tickets con resolución Done): 93.0%
Backlog Rate    (% tickets sin fecha de cierre): 1.3%

Interpretación:
  · Resolution Rate 93.0% > 85% → saludable ✓
  · Backlog Rate 1.3% → baja acumulación de trabajo pendiente ✓


In [6]:
# 3.4 · Diagnóstico de valores faltantes
# Las columnas wf_* tienen muchos nulos porque no todos los tickets pasan
# por todos los estados del workflow — esto es CORRECTO, no un error de calidad.
# Un nulo en wf_reopened significa que ese ticket nunca fue reabierto.

nulos = df_issues.isnull().sum()
pct   = (nulos / len(df_issues) * 100).round(2)
diag  = pd.DataFrame({'faltantes': nulos, 'pct_%': pct})
diag  = diag[diag['faltantes'] > 0].sort_values('pct_%', ascending=False)

print(f'Columnas con valores faltantes: {len(diag)} de {df_issues.shape[1]}')
print()
print(diag.to_string())
print()
print('Interpretación:')
print('  · wf_* con >90% nulos → estados poco frecuentes del workflow (esperado)')
print('  · issue_assignee ~46% nulos → tickets sin asignado formal')
print('  · issue_resolution_date ~1.3% nulos → backlog activo')

Columnas con valores faltantes: 24 de 58

                              faltantes  pct_%
wf_in_review                      66614  99.88
wf_rejected                       66579  99.83
wf_deployment                     66556  99.80
wf_cancelled                      66531  99.76
wf_pending_customer_approval      66437  99.62
wf_testing_monitoring             66401  99.57
wf_monitoring                     66173  99.22
wf_to_do                          65906  98.82
wf_done                           65831  98.71
wf_pending_deployment             65278  97.88
wf_closed                         65096  97.61
wf_approved                       65061  97.56
wf_under_review                   64870  97.27
wf_resolved_under_monitoring      64760  97.10
wf_reopened                       64568  96.82
wf_validation                     63730  95.56
wf_waiting                        46987  70.45
wf_resolved                       38546  57.80
issue_assignee                    31047  46.55
wf_in_progress    

In [7]:
# 3.5 · Rango temporal del dataset
fechas = pd.to_datetime(df_issues['issue_created'], format='ISO8601', utc=True)
print(f'Primer ticket:  {fechas.min().date()}')
print(f'Último ticket:  {fechas.max().date()}')
print(f'Cobertura:      {(fechas.max() - fechas.min()).days // 365} años')
print()
print('Ventana para análisis YoY: 2018–2022 (años con cobertura completa de 12 meses)')
print('Años 2016–2017: menor volumen y calidad de registro — sistema en adopción temprana')

Primer ticket:  2007-04-15
Último ticket:  2023-03-14
Cobertura:      15 años

Ventana para análisis YoY: 2018–2022 (años con cobertura completa de 12 meses)
Años 2016–2017: menor volumen y calidad de registro — sistema en adopción temprana


---
# PASO 4 — Limpieza y transformación

**Qué:** aplicar las decisiones de limpieza justificadas por el diagnóstico del Paso 3 y construir las variables derivadas necesarias para calcular los KPIs.  
**Principio:** cada decisión tiene una justificación de negocio documentada, no solo técnica.  
**Herramientas:** pandas · funciones Python

In [8]:
# 4.1 · Conversión de fechas
# format='ISO8601' maneja la variabilidad del formato original
# (con/sin microsegundos, con zona horaria UTC +00:00)

for col in ['issue_created', 'issue_resolution_date', 'last_change_date']:
    df_issues[col] = pd.to_datetime(df_issues[col], format='ISO8601', utc=True)

df_snapshot['started'] = pd.to_datetime(df_snapshot['started'], format='ISO8601', utc=True)
df_snapshot['ended']   = pd.to_datetime(df_snapshot['ended'],   format='ISO8601', utc=True)
df_history['created']  = pd.to_datetime(df_history['created'],  format='ISO8601', utc=True)

print('Fechas convertidas a datetime con zona horaria UTC.')

Fechas convertidas a datetime con zona horaria UTC.


In [9]:
# 4.2 · Filtro de prioridad 'unknown'
# ─────────────────────────────────────────────────────────────────────
# El 50.9% de los tickets tiene prioridad 'unknown' — son tickets que no
# pasaron por el flujo estándar de clasificación (resueltos informalmente,
# fuera del proceso, o sin clasificar en el sistema).
#
# Mezclarlos con los clasificados distorsionaría el MTTR por prioridad:
# sería como calcular el tiempo promedio de atención en urgencias
# mezclando emergencias reales con consultas administrativas.
# Se conservan en df_issues_raw como universo completo de referencia.

df_issues_raw = df_issues.copy()
df_issues     = df_issues[df_issues['issue_priority'] != 'unknown'].copy()

excluidos = len(df_issues_raw) - len(df_issues)
print(f'Tickets clasificados (universo de análisis):  {len(df_issues):>7,}')
print(f'Tickets unknown excluidos:                    {excluidos:>7,}  ({excluidos/len(df_issues_raw)*100:.1f}%)')

Tickets clasificados (universo de análisis):   32,726
Tickets unknown excluidos:                     33,965  (50.9%)


In [10]:
# 4.3 · Orden de severidad como tipo Categorical
# Permite que groupby y gráficos respeten el orden Blocker → Lowest
# sin código adicional en cada análisis

ORDEN_PRIORIDAD = ['Blocker', 'Highest', 'High', 'Medium', 'Low', 'Lowest']
df_issues['issue_priority'] = pd.Categorical(
    df_issues['issue_priority'], categories=ORDEN_PRIORIDAD, ordered=True
)
print('Orden de severidad definido:', ORDEN_PRIORIDAD)

Orden de severidad definido: ['Blocker', 'Highest', 'High', 'Medium', 'Low', 'Lowest']


In [11]:
# 4.4 · Clasificación de proyecto: cliente externo vs interno
# ─────────────────────────────────────────────────────────────────────
# Según FEATURES.md, los proyectos externos tienen códigos > 6 caracteres
# con formato [código_país][producto][cliente] — ej: C0378XPS.
# Los internos tienen códigos cortos — ej: d1z0, db6.
#
# La distinción es crítica desde el negocio: los proyectos externos tienen
# una relación comercial activa. Si su MTTR es mayor que el interno
# en la misma prioridad, el equipo está priorizando sus propios problemas
# sobre los del cliente — riesgo directo de pérdida de cuenta.

def clasificar_proyecto(codigo):
    if pd.isna(codigo): return 'sin_clasificar'
    return 'externo' if len(str(codigo)) > 6 else 'interno'

df_issues['tipo_proyecto'] = df_issues['issue_proj'].apply(clasificar_proyecto)

dist = df_issues['tipo_proyecto'].value_counts()
for tipo, n in dist.items():
    print(f'  {tipo:15} {n:>6,} tickets  ({n/len(df_issues)*100:.1f}%)')

  externo         22,001 tickets  (67.2%)
  interno         10,725 tickets  (32.8%)


In [12]:
# 4.5 · KPI: MTTR — Mean Time to Resolve (en horas)
# ─────────────────────────────────────────────────────────────────────
# MTTR es la métrica operativa más importante en helpdesk.
# Mide el tiempo real desde que el cliente reporta el problema
# hasta que el equipo lo resuelve — no el tiempo de procesamiento interno.
#
# Se calcula en horas para mayor granularidad.
# Solo se calcula para tickets con fecha de resolución.
# Los sin resolver quedan como NaN — incluirlos sesgaría el promedio hacia arriba.
#
# HALLAZGO ANTICIPADO: la media será mucho mayor que la mediana (outliers extremos).
# El KPI correcto para MTTR con distribuciones sesgadas es siempre la MEDIANA.

mask = df_issues['issue_resolution_date'].notna()
df_issues.loc[mask, 'mttr_horas'] = (
    df_issues.loc[mask, 'issue_resolution_date'] -
    df_issues.loc[mask, 'issue_created']
).dt.total_seconds() / 3600

df_issues['wf_total_horas'] = df_issues['wf_total_time'] / 3600

print(f'Tickets con MTTR calculado: {df_issues["mttr_horas"].notna().sum():,}')
print('Estadísticas de MTTR (horas):')
print(df_issues['mttr_horas'].describe().round(1).to_string())
print()
print('ATENCIÓN: media (1308.9h) >> mediana (194.9h) → distribución fuertemente sesgada.')
print('Outliers extremos detectados (máx 48,926h = 5.6 años).')
print('El KPI oficial de MTTR será la MEDIANA — robusta a outliers.')

Tickets con MTTR calculado: 31,873
Estadísticas de MTTR (horas):
count   31873.00
mean     1308.90
std      4064.30
min         0.00
25%        42.00
50%       194.90
75%       697.40
max     48926.60

ATENCIÓN: media (1308.9h) >> mediana (194.9h) → distribución fuertemente sesgada.
Outliers extremos detectados (máx 48,926h = 5.6 años).
El KPI oficial de MTTR será la MEDIANA — robusta a outliers.


In [13]:
# 4.6 · KPI: Ticket Churn (Reopen Rate)
# ─────────────────────────────────────────────────────────────────────
# Ticket Churn es el equivalente del churn de clientes aplicado a tickets:
# mide qué porcentaje de casos 'ganados' (cerrados) volvieron a perderse.
#
# Un ticket reabierto significa que el equipo declaró el problema resuelto
# pero el cliente no lo aceptó — tuvo que volver a abrirse.
# Es un indicador de CALIDAD de resolución, no de velocidad.
# Cada reapertura duplica el costo de atención del ticket.
#
# Base: wfe_reopened > 0 (número de veces que el ticket pasó por 'Reopened')

df_issues['fue_reabierto'] = df_issues['wfe_reopened'].fillna(0) > 0

ticket_churn = df_issues['fue_reabierto'].mean() * 100
print(f'Ticket Churn (Reopen Rate): {ticket_churn:.2f}%')
print(f'Tickets reabiertos:         {df_issues["fue_reabierto"].sum():,} de {len(df_issues):,}')

Ticket Churn (Reopen Rate): 2.56%
Tickets reabiertos:         839 de 32,726


In [14]:
# 4.7 · Variables temporales para análisis de Ticket Volume YoY

df_issues['anio']       = df_issues['issue_created'].dt.year
df_issues['mes']        = df_issues['issue_created'].dt.month
df_issues['dia_semana'] = df_issues['issue_created'].dt.dayofweek  # 0=lunes

print('Volumen de tickets por año (2018–2022, cobertura completa):')
vol = df_issues[df_issues['anio'].between(2018,2022)]['anio'].value_counts().sort_index()
for anio, n in vol.items():
    yoy = '—' if anio == 2018 else f'+{(n/vol[anio-1]-1)*100:.0f}%'
    print(f'  {anio}: {n:,} tickets  ({yoy})')
print(f'  Crecimiento total 2018→2022: +{(vol[2022]/vol[2018]-1)*100:.0f}%')

Volumen de tickets por año (2018–2022, cobertura completa):
  2018: 1,358 tickets  (—)
  2019: 4,797 tickets  (+253%)
  2020: 5,442 tickets  (+13%)
  2021: 5,832 tickets  (+7%)
  2022: 6,542 tickets  (+12%)
  Crecimiento total 2018→2022: +382%


---
# PASO 5 — Base de datos SQLite + queries de extracción SQL

**Qué:** montar las tres tablas en SQLite (réplica del PostgreSQL original) y ejecutar 4 queries SQL para extraer métricas con lógica relacional.  

**ETL vs ELT en la práctica:**
- En este notebook: **ETL** — Python transforma y luego carga el resultado limpio.
- En Databricks/BigQuery: **ELT** — se carga el dato crudo primero, se transforma con SQL/Spark dentro de la plataforma. El resultado es el mismo; la diferencia es dónde ocurre la transformación.

**Por qué SQL separado del código Python:** en equipos de datos reales, DBA y analistas trabajan en capas distintas. El archivo `.sql` puede ser revisado, versionado y ejecutado independientemente.  
**Herramientas:** sqlite3 · SQL (INNER JOIN · CASE · JULIANDAY · SUBSTR · GROUP BY)

In [15]:
# Crear base de datos en memoria y cargar las tres tablas relacionales
conn = sqlite3.connect(':memory:')

# Se carga el dataset RAW (con unknown incluido) para que las queries filtren ellas mismas
df_issues_raw.to_sql('issues',             conn, if_exists='replace', index=False)
df_history.to_sql('issues_change_history', conn, if_exists='replace', index=False)
df_snapshot.to_sql('issues_snapshot',      conn, if_exists='replace', index=False)

print('Base de datos SQLite en memoria — tablas cargadas:')
for t in ['issues', 'issues_change_history', 'issues_snapshot']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0, 0]
    print(f'  {t:<30} {n:>7,} filas')

Base de datos SQLite en memoria — tablas cargadas:
  issues                          66,691 filas
  issues_change_history          257,508 filas
  issues_snapshot                 90,963 filas


In [16]:
# Query SQL 1 — MTTR por prioridad
# JULIANDAY() convierte timestamps a días decimales en SQLite; × 24 = horas
# SQLite no tiene PERCENTILE_CONT — la mediana se calcula en pandas (Paso 6)

query_mttr = """
SELECT
    issue_priority,
    COUNT(*) AS total_tickets,
    ROUND(AVG((JULIANDAY(issue_resolution_date) - JULIANDAY(issue_created)) * 24), 1)
        AS mttr_promedio_horas,
    ROUND(MIN((JULIANDAY(issue_resolution_date) - JULIANDAY(issue_created)) * 24), 1)
        AS mttr_minimo_horas,
    ROUND(MAX((JULIANDAY(issue_resolution_date) - JULIANDAY(issue_created)) * 24), 1)
        AS mttr_maximo_horas
FROM issues
WHERE issue_resolution_date IS NOT NULL
  AND issue_created         IS NOT NULL
  AND issue_priority NOT IN ('unknown')
GROUP BY issue_priority
ORDER BY total_tickets DESC
"""

df_mttr_sql = pd.read_sql(query_mttr, conn)
print('MTTR por prioridad (extraído con SQL):')
print(df_mttr_sql.to_string(index=False))
print()
print('NOTA: el promedio está distorsionado por outliers extremos.')
print('La mediana (calculada en Paso 6 con pandas) es el KPI correcto.')

MTTR por prioridad (extraído con SQL):
issue_priority  total_tickets  mttr_promedio_horas  mttr_minimo_horas  mttr_maximo_horas
        Medium          24249              1515.50               0.00           48926.60
          High           4381               591.20               0.00           40255.50
       Highest           2018               485.90               0.00           36717.60
       Blocker            641              1148.20               0.00           42164.20
           Low            501              1107.50               0.00           29418.00
        Lowest             83              1304.80               0.00           24575.90

NOTA: el promedio está distorsionado por outliers extremos.
La mediana (calculada en Paso 6 con pandas) es el KPI correcto.


In [17]:
# Query SQL 2 — FCR y Escalation Rate desde el historial de cambios
# ─────────────────────────────────────────────────────────────────────
# FCR (First Contact Resolution): % de tickets resueltos sin reasignación
#   Un ticket tiene FCR cuando el campo 'assignee' no cambió más de una vez.
#
# Escalation Rate: % de tickets donde 'assignee' cambió más de una vez
#   Cada reasignación adicional = un nivel de escalación = más horas de MTTR.
#
# IMPORTANTE: esta información solo existe en issues_change_history.
# El INNER JOIN entre las dos tablas es lo que hace posible este cálculo.
# Sin el JOIN, FCR y Escalation Rate son imposibles de calcular con precisión.

query_fcr_escal = """
SELECT
    ch.issueid,
    i.issue_priority,
    i.issue_type,
    SUM(CASE WHEN ch.field = 'assignee' THEN 1 ELSE 0 END) AS num_reasignaciones,
    SUM(CASE WHEN ch.field = 'status'   THEN 1 ELSE 0 END) AS num_cambios_estado,
    CASE
        WHEN SUM(CASE WHEN ch.field = 'assignee' THEN 1 ELSE 0 END) <= 1
        THEN 'fcr'
        ELSE 'escalado'
    END AS tipo_resolucion
FROM issues_change_history ch
    INNER JOIN issues i ON ch.issueid = i.id
WHERE i.issue_priority NOT IN ('unknown')
GROUP BY ch.issueid, i.issue_priority, i.issue_type
"""

df_escal_sql = pd.read_sql(query_fcr_escal, conn)

fcr_rate   = (df_escal_sql['tipo_resolucion'] == 'fcr').mean() * 100
escal_rate = (df_escal_sql['tipo_resolucion'] == 'escalado').mean() * 100

print(f'Tickets analizados:        {len(df_escal_sql):,}')
print(f'FCR (First Contact Res.):  {fcr_rate:.1f}%')
print(f'Escalation Rate:           {escal_rate:.1f}%')
print()
print('Distribución:')
print(df_escal_sql['tipo_resolucion'].value_counts().to_string())

Tickets analizados:        32,686
FCR (First Contact Res.):  57.1%
Escalation Rate:           42.9%

Distribución:
tipo_resolucion
fcr         18655
escalado    14031


In [18]:
# Query SQL 3 — Tendencia mensual por prioridad (2018–2022)
# SUBSTR extrae año-mes del timestamp como texto (ej: '2020-03')
# Ventana 2018–2022: años con cobertura completa de 12 meses

query_tendencia = """
SELECT
    SUBSTR(issue_created, 1, 7) AS mes,
    issue_priority,
    COUNT(*) AS tickets_creados,
    SUM(CASE WHEN issue_resolution = 'Done' THEN 1 ELSE 0 END) AS tickets_resueltos,
    ROUND(
        SUM(CASE WHEN issue_resolution = 'Done' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1
    ) AS resolution_rate_mensual
FROM issues
WHERE issue_priority NOT IN ('unknown')
  AND issue_created >= '2018-01-01'
  AND issue_created <  '2023-01-01'
GROUP BY mes, issue_priority
ORDER BY mes, issue_priority
"""

df_tendencia_sql = pd.read_sql(query_tendencia, conn)
print(f'Meses en el análisis: {df_tendencia_sql["mes"].nunique()}')
print(df_tendencia_sql.head(4).to_string(index=False))

Meses en el análisis: 60
    mes issue_priority  tickets_creados  tickets_resueltos  resolution_rate_mensual
2018-01           High                4                  4                   100.00
2018-01        Highest                8                  8                   100.00
2018-01            Low                1                  1                   100.00
2018-01         Medium              123                121                    98.40


In [19]:
# Query SQL 4 — Carga por turno desde issues_snapshot
# 'turn=1' = primer asignado (sin escalación)
# 'turn>1' = escalaciones — mide el costo en horas de cada nivel adicional

query_turnos = """
SELECT
    turn AS turno_asignacion,
    issue_priority,
    COUNT(*) AS registros,
    ROUND(AVG((JULIANDAY(ended) - JULIANDAY(started)) * 24), 1)
        AS horas_promedio_por_turno
FROM issues_snapshot
WHERE issue_priority NOT IN ('unknown')
  AND started IS NOT NULL AND ended IS NOT NULL AND ended > started
GROUP BY turn, issue_priority
ORDER BY turn, issue_priority
"""

df_turnos_sql = pd.read_sql(query_turnos, conn)
print('Horas promedio por turno de asignación (primeras 12 filas):')
print(df_turnos_sql.head(12).to_string(index=False))

Horas promedio por turno de asignación (primeras 12 filas):
 turno_asignacion issue_priority  registros  horas_promedio_por_turno
                1        Blocker        656                    703.60
                1           High       4547                    230.00
                1        Highest       2083                    170.70
                1            Low        558                    813.30
                1         Lowest         84                   1247.70
                1         Medium      24721                   1292.10
                2        Blocker        463                    366.50
                2           High       2912                    367.00
                2        Highest       1501                    324.00
                2            Low        176                    738.80
                2         Lowest         19                    444.80
                2         Medium       8603                    529.40


---
# PASO 6 — KPIs completos en pandas + exportación

**Qué:** calcular medianas, percentiles, SLA Compliance Rate y Avg Contributors que SQLite no soporta. Exportar los 6 datasets limpios para el Notebook 2 y Power BI.  
**Por qué pandas aquí y no SQL:** SQLite no tiene `PERCENTILE_CONT` ni `MEDIAN`. Las medianas son esenciales para MTTR en distribuciones sesgadas como esta.  
**Herramienta:** pandas · `.groupby()` · `.agg()` · `.to_csv()`

In [20]:
# MTTR completo con percentiles por prioridad

df_mttr_completo = (
    df_issues[df_issues['mttr_horas'].notna()]
    .groupby('issue_priority', observed=True)['mttr_horas']
    .agg(
        n       = 'count',
        mediana = 'median',
        media   = 'mean',
        p25     = lambda x: x.quantile(0.25),
        p75     = lambda x: x.quantile(0.75)
    )
    .round(1).reset_index()
)

# Agregar días para legibilidad ejecutiva
df_mttr_completo['mediana_dias'] = (df_mttr_completo['mediana'] / 24).round(1)

print('MTTR por prioridad con percentiles:')
print(df_mttr_completo.to_string(index=False))
print()
print('Hallazgo: media >> mediana en todos los niveles — outliers extremos presentes.')
print('El KPI oficial es la MEDIANA. La media se documenta pero no se usa como indicador.')

MTTR por prioridad con percentiles:
issue_priority     n  mediana   media   p25    p75  mediana_dias
       Blocker   641    90.60 1148.20 17.10 477.70          3.80
       Highest  2018    91.90  485.90  3.90 338.30          3.80
          High  4381   120.50  591.20 21.60 459.80          5.00
        Medium 24249   238.50 1515.50 52.30 816.00          9.90
           Low   501   216.60 1107.50 27.20 794.80          9.00
        Lowest    83   230.00 1304.80 49.50 907.30          9.60

Hallazgo: media >> mediana en todos los niveles — outliers extremos presentes.
El KPI oficial es la MEDIANA. La media se documenta pero no se usa como indicador.


In [21]:
# Ticket Churn (Reopen Rate) por prioridad
# ─────────────────────────────────────────────────────────────────────
# Se espera que el Ticket Churn sea mayor en prioridades altas
# (más presión para cerrar rápido = más riesgo de cierre apresurado).
# Si el patrón es inverso, sugiere problemas de calidad en prioridades bajas.

df_churn = (
    df_issues
    .groupby('issue_priority', observed=True)
    .agg(total=('id','count'), reabiertos=('fue_reabierto','sum'))
    .assign(ticket_churn_pct=lambda x: (x['reabiertos']/x['total']*100).round(2))
    .reset_index()
)

print('Ticket Churn (Reopen Rate) por prioridad:')
print(df_churn.to_string(index=False))
print()
print('Hallazgo: Low (4.82%) y Highest (4.08%) tienen mayor churn que Blocker (2.29%).')
print('Los tickets críticos se cierran con más cuidado.')
print('Los de baja/media prioridad se cierran apresurados — y vuelven.')

Ticket Churn (Reopen Rate) por prioridad:
issue_priority  total  reabiertos  ticket_churn_pct
       Blocker    656          15              2.29
       Highest   2084          85              4.08
          High   4554         166              3.65
        Medium  24788         542              2.19
           Low    560          27              4.82
        Lowest     84           4              4.76

Hallazgo: Low (4.82%) y Highest (4.08%) tienen mayor churn que Blocker (2.29%).
Los tickets críticos se cierran con más cuidado.
Los de baja/media prioridad se cierran apresurados — y vuelven.


In [22]:
# SLA Compliance Rate — proxy
# ─────────────────────────────────────────────────────────────────────
# SLA Compliance Rate: % de tickets resueltos dentro del umbral de tiempo acordado.
# En ITSM, el incumplimiento de SLA puede tener penalidades contractuales.
#
# Como no hay SLA formal en el dataset, se usan umbrales de referencia estándar:
#   · Blocker:  8 horas (problema crítico — impacto en producción)
#   · Highest: 24 horas (problema severo)
#   · High:    48 horas (problema importante)
#   · Medium: 120 horas (problema moderado — 5 días hábiles)

sla_umbrales = {'Blocker': 8, 'Highest': 24, 'High': 48, 'Medium': 120}

print('SLA Compliance Rate por prioridad (proxy con umbrales estándar):')
print(f'{"Prioridad":<12} {"Umbral":<12} {"Dentro del SLA":<18} {"Cumplimiento"}')
print('-' * 55)
for prioridad, umbral in sla_umbrales.items():
    sub       = df_issues[(df_issues['issue_priority'] == prioridad) & df_issues['mttr_horas'].notna()]
    dentro    = (sub['mttr_horas'] <= umbral).sum()
    total     = len(sub)
    pct       = dentro / total * 100 if total > 0 else 0
    alerta    = '⚠️  CRÍTICO' if pct < 30 else ('⚠️' if pct < 60 else '✓')
    print(f'{prioridad:<12} < {umbral:<9}h  {dentro:>5,} de {total:>5,}     {pct:>5.1f}%  {alerta}')

print()
print('HALLAZGO CRÍTICO: Blocker SLA compliance 22.5% — solo 1 de cada 4 tickets críticos')
print('se resuelve dentro del umbral de 8 horas. Esto tiene impacto directo en contratos.')

SLA Compliance Rate por prioridad (proxy con umbrales estándar):
Prioridad    Umbral       Dentro del SLA     Cumplimiento
-------------------------------------------------------
Blocker      < 8        h    144 de   641      22.5%  ⚠️  CRÍTICO
Highest      < 24       h    702 de 2,018      34.8%  ⚠️
High         < 48       h  1,539 de 4,381      35.1%  ⚠️
Medium       < 120      h  8,614 de 24,249      35.5%  ⚠️

HALLAZGO CRÍTICO: Blocker SLA compliance 22.5% — solo 1 de cada 4 tickets críticos
se resuelve dentro del umbral de 8 horas. Esto tiene impacto directo en contratos.


In [23]:
# Avg Contributors per Ticket por prioridad
# ─────────────────────────────────────────────────────────────────────
# Avg Contributors = promedio de personas involucradas por ticket.
# Un valor > 1 indica que múltiples ingenieros trabajaron en el mismo problema.
# Blocker con avg 1.89 significa que casi todos los tickets críticos
# requieren al menos dos personas — lo cual eleva el costo y el MTTR.

df_contribs = (
    df_issues
    .groupby('issue_priority', observed=True)['issue_contr_count']
    .agg(media='mean', mediana='median', max_='max')
    .round(2).reset_index()
)

print('Avg Contributors per Ticket por prioridad:')
print(df_contribs.to_string(index=False))

Avg Contributors per Ticket por prioridad:
issue_priority  media  mediana  max_
       Blocker   1.89     2.00  5.00
       Highest   1.85     2.00  7.00
          High   1.77     2.00  8.00
        Medium   1.30     1.00 10.00
           Low   1.36     1.00  4.00
        Lowest   1.19     1.00  2.00


In [24]:
# Exportación de todos los datasets limpios
# CSV: formato más compatible entre Python, Power BI y SQL

cols = [
    'id','issue_proj','issue_priority','issue_type',
    'issue_created','issue_resolution_date','issue_resolution','issue_status',
    'issue_contr_count','issue_comments_count','wf_total_horas','mttr_horas',
    'processing_steps','fue_reabierto','tipo_proyecto','anio','mes','dia_semana',
    'wfe_reopened','wfe_in_progress','wfe_resolved','wfe_waiting'
]
df_main = df_issues[[c for c in cols if c in df_issues.columns]].copy()

for col in ['issue_created','issue_resolution_date']:
    if col in df_main.columns:
        df_main[col] = df_main[col].dt.strftime('%Y-%m-%d %H:%M:%S')

exports = [
    ('issues_limpio.csv',        df_main,          'Dataset consolidado para Power BI'),
    ('escalaciones.csv',         df_escal_sql,     'FCR y Escalation Rate por ticket (JOIN SQL)'),
    ('tendencia_mensual.csv',    df_tendencia_sql, 'Volumen y Resolution Rate mensual'),
    ('mttr_por_prioridad.csv',   df_mttr_completo, 'MTTR estadístico completo con percentiles'),
    ('ticket_churn.csv',         df_churn,         'Reopen Rate por prioridad'),
    ('carga_por_turno.csv',      df_turnos_sql,    'Horas por turno de asignación'),
]

print('Archivos exportados a exports/:')
for nombre, df_, desc in exports:
    df_.to_csv(RUTA_EXPORTS + nombre, index=False)
    print(f'  {nombre:<35} {len(df_):>7,} filas  — {desc}')

Archivos exportados a exports/:
  issues_limpio.csv                    32,726 filas  — Dataset consolidado para Power BI
  escalaciones.csv                     32,686 filas  — FCR y Escalation Rate por ticket (JOIN SQL)
  tendencia_mensual.csv                   293 filas  — Volumen y Resolution Rate mensual
  mttr_por_prioridad.csv                    6 filas  — MTTR estadístico completo con percentiles
  ticket_churn.csv                          6 filas  — Reopen Rate por prioridad
  carga_por_turno.csv                      57 filas  — Horas por turno de asignación


---
## Resumen del Notebook 1

| Paso | Qué se hizo | Herramienta | Output |
|---|---|---|---|
| 1 | Importaciones y configuración de rutas | Python | Entorno reproducible |
| 2 | Carga de 3 archivos CSV relacionales | pandas | 3 DataFrames |
| 3 | Exploración: duplicados · nulos · distribución · Resolution Rate · Backlog | pandas | Diagnóstico de calidad |
| 4 | Limpieza: fechas · filtro unknown · clasificación · MTTR · Ticket Churn · tiempo | pandas | Dataset limpio con KPIs base |
| 5 | SQLite + 4 queries SQL con JOINs entre tablas | SQL · sqlite3 | FCR · Escalation · MTTR SQL · Tendencia |
| 6 | MTTR mediana · Ticket Churn · SLA Compliance · Avg Contributors · Exportación | pandas | 6 CSVs para Notebook 2 y Power BI |

**Hallazgos clave de este notebook:**

| KPI | Valor | Señal |
|---|---|---|
| MTTR mediana Blocker | 90.6 h (3.8 días) | Aceptable pero con margen de mejora |
| FCR | 57.1% | Solo 57 de cada 100 tickets se resuelven sin escalar |
| Escalation Rate | 42.9% | Casi la mitad de los tickets escala a otro nivel |
| Ticket Churn | 2.56% global | Low y Highest tienen mayor churn que Blocker |
| SLA Compliance Blocker | 22.5% | **CRÍTICO** — 3 de cada 4 Blockers incumplen SLA |
| Ticket Volume | +382% en 4 años | Crecimiento que requiere planificación de capacidad |

---
**Siguiente → `02_analisis_operativo.ipynb`**  
Validación estadística · visualizaciones · segmentación de riesgo · conclusiones de negocio